# Bank Churners Dataset Analysis

This notebook performs cleaning and classification using Logistic Regression, Random Forest, and XGBoost.

In [1]:

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix

# Load data
df = pd.read_csv("BankChurners.csv")

# Drop unnecessary columns
df_cleaned = df.drop(columns=["CLIENTNUM"] + [col for col in df.columns if "Naive_Bayes" in col])
df_cleaned = df_cleaned.dropna()

# Encode categorical variables
label_encoders = {}
for col in df_cleaned.select_dtypes(include="object").columns:
    le = LabelEncoder()
    df_cleaned[col] = le.fit_transform(df_cleaned[col])
    label_encoders[col] = le

# Define features and target
X = df_cleaned.drop(columns=["Attrition_Flag"])
y = df_cleaned["Attrition_Flag"]

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [2]:

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

print("Logistic Regression")
print(confusion_matrix(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))


Logistic Regression
[[ 158  169]
 [  59 1640]]
              precision    recall  f1-score   support

           0       0.73      0.48      0.58       327
           1       0.91      0.97      0.94      1699

    accuracy                           0.89      2026
   macro avg       0.82      0.72      0.76      2026
weighted avg       0.88      0.89      0.88      2026



c:\ProgramData\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [3]:

lr = LogisticRegression(max_iter=5000)
lr.fit(X_train_scaled, y_train)
y_pred_lr = lr.predict(X_test_scaled)

print("Logistic Regression")
print(confusion_matrix(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))


Logistic Regression
[[ 178  149]
 [  52 1647]]
              precision    recall  f1-score   support

           0       0.77      0.54      0.64       327
           1       0.92      0.97      0.94      1699

    accuracy                           0.90      2026
   macro avg       0.85      0.76      0.79      2026
weighted avg       0.89      0.90      0.89      2026



In [4]:
coefs = np.abs(lr.coef_[0])
lr_importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": coefs
}).sort_values(by="Importance", ascending=False)
print("Logistic Regression Feature Importance:\n", lr_importance)

NameError: name 'feature_names' is not defined

In [ ]:

rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print("Random Forest Classifier")
print(confusion_matrix(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))


Random Forest Classifier
[[ 266   61]
 [  23 1676]]
              precision    recall  f1-score   support

           0       0.92      0.81      0.86       327
           1       0.96      0.99      0.98      1699

    accuracy                           0.96      2026
   macro avg       0.94      0.90      0.92      2026
weighted avg       0.96      0.96      0.96      2026



In [ ]:
importances = rf.feature_importances_
rf_importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
}).sort_values(by="Importance", ascending=False)
print("Random Forest Feature Importance:\n", rf_importance)

Random Forest Feature Importance:
                      Feature  Importance
15           Total_Trans_Amt    0.192932
16            Total_Trans_Ct    0.167267
17       Total_Ct_Chng_Q4_Q1    0.113093
12       Total_Revolving_Bal    0.097576
18     Avg_Utilization_Ratio    0.067393
8   Total_Relationship_Count    0.066600
14      Total_Amt_Chng_Q4_Q1    0.062215
11              Credit_Limit    0.034976
0               Customer_Age    0.031849
13           Avg_Open_To_Buy    0.031151
9     Months_Inactive_12_mon    0.026432
10     Contacts_Count_12_mon    0.026223
7             Months_on_book    0.025469
2            Dependent_count    0.013079
3            Education_Level    0.011160
1                     Gender    0.010927
5            Income_Category    0.010869
4             Marital_Status    0.008767
6              Card_Category    0.002021


In [ ]:

xgb = XGBClassifier(eval_metric='mlogloss')
xgb.fit(X_train, y_train)
y_pred_xgb = xgb.predict(X_test)

print("XGBoost Classifier")
print(confusion_matrix(y_test, y_pred_xgb))
print(classification_report(y_test, y_pred_xgb))


XGBoost Classifier
[[ 288   39]
 [  34 1665]]
              precision    recall  f1-score   support

           0       0.89      0.88      0.89       327
           1       0.98      0.98      0.98      1699

    accuracy                           0.96      2026
   macro avg       0.94      0.93      0.93      2026
weighted avg       0.96      0.96      0.96      2026



In [ ]:
# Get feature importances
importances = xgb.feature_importances_
feature_names = X_train.columns
importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
}).sort_values(by="Importance", ascending=False)

# Display top features
print(importance_df)

                     Feature  Importance
16            Total_Trans_Ct    0.249950
12       Total_Revolving_Bal    0.200179
8   Total_Relationship_Count    0.120059
15           Total_Trans_Amt    0.067791
17       Total_Ct_Chng_Q4_Q1    0.050848
9     Months_Inactive_12_mon    0.045586
0               Customer_Age    0.043511
10     Contacts_Count_12_mon    0.035982
1                     Gender    0.031794
14      Total_Amt_Chng_Q4_Q1    0.030429
11              Credit_Limit    0.028467
13           Avg_Open_To_Buy    0.020446
18     Avg_Utilization_Ratio    0.017390
7             Months_on_book    0.013313
2            Dependent_count    0.011611
4             Marital_Status    0.010923
5            Income_Category    0.008803
3            Education_Level    0.008800
6              Card_Category    0.004118
